# Notebook 07 — Penentuan Bobot Kriteria dengan AHP

**Tujuan:** Menentukan **bobot 8 kriteria** SPK secara terstruktur menggunakan
*Analytic Hierarchy Process* (AHP), menggantikan bobot ilustrasi di NB06 dengan bobot
yang dapat dipertanggungjawabkan dan **teruji konsistensinya**.

**Alur AHP:**
1. Susun **matriks perbandingan berpasangan** antar kriteria (skala Saaty 1–9)
2. Hitung **vektor prioritas** (bobot) dari matriks tersebut
3. Uji **rasio konsistensi (CR)** — hasil sahih jika CR < 0,10
4. Terapkan bobot AHP untuk menghitung ulang ranking TOPSIS & SAW

> **PENTING (kejujuran ilmiah):** nilai perbandingan berpasangan di bawah adalah
> **template awal** berdasarkan logika penargetan kemiskinan (Desil 1 paling penting, dst.).
> Untuk penelitian final, nilai-nilai ini **wajib divalidasi/diisi oleh ahli** — idealnya
> petugas Dinas Sosial Kab. Lahat melalui wawancara — agar bobot benar-benar sah.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path('../').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from utils import RESULTS_DIR

SPK_DIR = ROOT / 'output' / 'spk'
SPK_DIR.mkdir(parents=True, exist_ok=True)

# CATATAN REVISI: kriteria 'pct_nonaktif' DIKELUARKAN dari pembobotan.
# Status nonaktif dapat terjadi karena sebab wajar (keluarga sudah mampu,
# pindah domisili, atau meninggal), sehingga tidak selalu menandakan
# kebutuhan intervensi. Kriteria ini tetap dicatat sebagai data pendukung,
# tetapi tidak diberi bobot dalam perhitungan prioritas.
KRITERIA = ['pct_desil1', 'pct_desil2', 'pct_desil3', 'pct_desil4',
            'pct_desil5', 'pct_desil6_10', 'pct_belum_prmk']
LABEL = ['Desil1', 'Desil2', 'Desil3', 'Desil4', 'Desil5', 'Desil6-10', 'BelumPRMK']
n = len(KRITERIA)
print('Jumlah kriteria:', n, '(Nonaktif dikeluarkan dari pembobotan)')

Jumlah kriteria: 7 (Nonaktif dikeluarkan dari pembobotan)


## 1. Skala Saaty (1–9)

| Nilai | Arti |
|---|---|
| 1 | Sama penting |
| 3 | Sedikit lebih penting |
| 5 | Lebih penting |
| 7 | Sangat lebih penting |
| 9 | Mutlak lebih penting |
| 2,4,6,8 | Nilai antara |
| 1/x | Kebalikan (kriteria baris kurang penting dari kolom) |

## 2. Matriks Perbandingan Berpasangan

Baris dibandingkan terhadap kolom. Contoh: Desil 1 vs Desil 5 = 5 berarti Desil 1
"lebih penting" daripada Desil 5. Urutan kepentingan yang diterapkan:
**Desil 1 > Desil 2 = Belum-PRMK > Desil 3 = Desil 6-10 > Desil 4 > Desil 5.**

**Revisi:** kriteria *Nonaktif* dikeluarkan dari matriks (tidak diberi bobot), karena status
nonaktif dapat disebabkan hal wajar (keluarga sudah mampu, pindah, atau meninggal) sehingga
tidak selalu merepresentasikan kebutuhan intervensi. Jumlah kriteria menjadi **7**.

Catatan bobot *Desil 6-10*: kolom ini merupakan **agregat 5 desil**, sehingga bobot per desil
sebenarnya paling kecil (0,0965 / 5 ≈ 0,019) — logika monotonik tetap terjaga.

In [2]:
# Matriks perbandingan berpasangan (skala Saaty), 7 kriteria.
# Diagonal = 1; segitiga bawah = kebalikan. Kriteria 'Nonaktif' DIKELUARKAN.
# Urutan: d1, d2, d3, d4, d5, d6_10, belum
A = np.array([
    [1,   2,   3,   4,   5,   3,   2  ],   # Desil 1
    [1/2, 1,   2,   3,   4,   2,   1  ],   # Desil 2
    [1/3, 1/2, 1,   2,   3,   1,   1/2],   # Desil 3
    [1/4, 1/3, 1/2, 1,   2,   1/2, 1/3],   # Desil 4
    [1/5, 1/4, 1/3, 1/2, 1,   1/3, 1/4],   # Desil 5
    [1/3, 1/2, 1,   2,   3,   1,   1/2],   # Desil 6-10
    [1/2, 1,   2,   3,   4,   2,   1  ],   # Belum PRMK
], dtype=float)

df_A = pd.DataFrame(A.round(3), index=LABEL, columns=LABEL)
print('Matriks perbandingan berpasangan (Saaty), 7 kriteria:')
df_A

Matriks perbandingan berpasangan (Saaty), 7 kriteria:


,Desil1,Desil2,Desil3,Desil4,Desil5,Desil6-10,BelumPRMK
Desil1,1.000,2.000,3.000,4.0,5.0,3.000,2.000
Desil2,0.500,1.000,2.000,3.0,4.0,2.000,1.000
Desil3,0.333,0.500,1.000,2.0,3.0,1.000,0.500
Desil4,0.250,0.333,0.500,1.0,2.0,0.500,0.333
Desil5,0.200,0.250,0.333,0.5,1.0,0.333,0.250
Desil6-10,0.333,0.500,1.000,2.0,3.0,1.000,0.500
BelumPRMK,0.500,1.000,2.000,3.0,4.0,2.000,1.000


## 3. Hitung Vektor Prioritas (Bobot)

Metode rata-rata kolom ternormalisasi: setiap kolom dibagi jumlahnya, lalu tiap baris
dirata-ratakan → itulah bobot kriteria.

In [3]:
col_sum = A.sum(axis=0)
A_norm = A / col_sum                       # normalisasi kolom
w = A_norm.mean(axis=1)                     # rata-rata baris = bobot

bobot = pd.DataFrame({'kriteria': KRITERIA, 'label': LABEL, 'bobot_AHP': w.round(4)})
bobot = bobot.sort_values('bobot_AHP', ascending=False).reset_index(drop=True)
print('Total bobot:', round(w.sum(), 6))
bobot

Total bobot: 1.0


,kriteria,label,bobot_AHP
0,pct_desil1,Desil1,0.3047
1,pct_desil2,Desil2,0.1858
2,pct_belum_prmk,BelumPRMK,0.1858
3,pct_desil3,Desil3,0.1078
4,pct_desil6_10,Desil6-10,0.1078
5,pct_desil4,Desil4,0.0652
6,pct_desil5,Desil5,0.0427


## 4. Uji Konsistensi (Consistency Ratio)

`λmax` = rata-rata dari (A·w)/w. `CI = (λmax − n)/(n − 1)`. `CR = CI / RI`, dengan RI =
Random Index (untuk n=8, RI = 1,41). **Hasil dianggap konsisten (sahih) bila CR < 0,10.**

In [4]:
Aw = A @ w
lamda_max = (Aw / w).mean()
CI = (lamda_max - n) / (n - 1)
RI_TABLE = {1:0, 2:0, 3:0.58, 4:0.90, 5:1.12, 6:1.24, 7:1.32, 8:1.41, 9:1.45, 10:1.49}
RI = RI_TABLE[n]
CR = CI / RI

print(f'lambda_max : {lamda_max:.4f}')
print(f'CI         : {CI:.4f}')
print(f'RI (n={n})  : {RI}')
print(f'CR         : {CR:.4f}')
print()
if CR < 0.10:
    print(f'>>> CR = {CR:.4f} < 0,10  →  MATRIKS KONSISTEN, bobot SAHIH digunakan.')
else:
    print(f'>>> CR = {CR:.4f} >= 0,10 →  perlu revisi perbandingan berpasangan.')

lambda_max : 7.0789
CI         : 0.0131
RI (n=7)  : 1.32
CR         : 0.0100

>>> CR = 0.0100 < 0,10  →  MATRIKS KONSISTEN, bobot SAHIH digunakan.


## 5. Perbandingan: Bobot AHP vs Bobot Ilustrasi (NB06)

In [5]:
# Bobot ilustrasi awal (NB06) dinormalisasi ulang tanpa Nonaktif, agar setara dibandingkan
bobot_ilustrasi_8 = {'pct_desil1':0.25,'pct_desil2':0.20,'pct_desil3':0.10,'pct_desil4':0.07,
                     'pct_desil5':0.05,'pct_desil6_10':0.08,'pct_belum_prmk':0.15}
tot = sum(bobot_ilustrasi_8.values())
bobot_ilustrasi = {k: v/tot for k, v in bobot_ilustrasi_8.items()}

banding = pd.DataFrame({
    'kriteria': KRITERIA,
    'bobot_ilustrasi_dinormalisasi': [round(bobot_ilustrasi[c], 4) for c in KRITERIA],
    'bobot_AHP': w.round(4),
})
banding['bobot_AHP_%'] = (banding['bobot_AHP'] * 100).round(1)
print(banding.to_string(index=False))

      kriteria  bobot_ilustrasi_dinormalisasi  bobot_AHP  bobot_AHP_%
    pct_desil1                         0.2778     0.3047         30.5
    pct_desil2                         0.2222     0.1858         18.6
    pct_desil3                         0.1111     0.1078         10.8
    pct_desil4                         0.0778     0.0652          6.5
    pct_desil5                         0.0556     0.0427          4.3
 pct_desil6_10                         0.0889     0.1078         10.8
pct_belum_prmk                         0.1667     0.1858         18.6


## 6. Terapkan Bobot AHP → Ranking TOPSIS & SAW

Bobot AHP dipakai ulang untuk menghitung skor prioritas 371 desa, lalu dibandingkan
peringkatnya dengan hasil bobot ilustrasi di NB06.

In [6]:
from scipy.stats import spearmanr

df = pd.read_csv(RESULTS_DIR / 'dtsen_clean.csv')
km = pd.read_csv(RESULTS_DIR / 'hasil_kmeans.csv')[['KECAMATAN','KELURAHAN','cluster_kmeans']]
df = df.merge(km, on=['KECAMATAN','KELURAHAN'], how='left')

X = df[KRITERIA].values.astype(float)
IS_BEN = np.array([c != 'pct_desil6_10' for c in KRITERIA])   # desil6-10 = cost
eps = 1e-9

# TOPSIS dengan bobot AHP
R = X / (np.sqrt((X**2).sum(axis=0)) + eps)
V = R * w
A_pos = np.where(IS_BEN, V.max(axis=0), V.min(axis=0))
A_neg = np.where(IS_BEN, V.min(axis=0), V.max(axis=0))
Dp = np.sqrt(((V-A_pos)**2).sum(axis=1)); Dn = np.sqrt(((V-A_neg)**2).sum(axis=1))
df['skor_topsis_ahp'] = (Dn/(Dp+Dn+eps)).round(6)
df['rank_topsis_ahp'] = df['skor_topsis_ahp'].rank(ascending=False, method='min').astype(int)

# SAW dengan bobot AHP
Rs = np.zeros_like(X)
for j in range(X.shape[1]):
    col = X[:,j]
    Rs[:,j] = col/(col.max()+eps) if IS_BEN[j] else (col.min()+eps)/(col+eps)
df['skor_saw_ahp'] = (Rs*w).sum(axis=1).round(6)
df['rank_saw_ahp'] = df['skor_saw_ahp'].rank(ascending=False, method='min').astype(int)

# bandingkan TOPSIS-AHP vs TOPSIS-ilustrasi (dari NB06)
prev = pd.read_csv(SPK_DIR / 'ranking_prioritas_desa.csv')[['KECAMATAN','KELURAHAN','rank_topsis']]
df = df.merge(prev, on=['KECAMATAN','KELURAHAN'], how='left')
rho_bobot, _ = spearmanr(df['rank_topsis_ahp'], df['rank_topsis'])
rho_metode, _ = spearmanr(df['rank_topsis_ahp'], df['rank_saw_ahp'])

print(f'Korelasi ranking TOPSIS: bobot AHP vs ilustrasi = {rho_bobot:.4f}')
print(f'Korelasi ranking (bobot AHP): TOPSIS vs SAW      = {rho_metode:.4f}')

Korelasi ranking TOPSIS: bobot AHP vs ilustrasi = 0.9885
Korelasi ranking (bobot AHP): TOPSIS vs SAW      = 0.9339


In [7]:
out = df.sort_values('rank_topsis_ahp')
show = ['KECAMATAN','KELURAHAN','cluster_kmeans','pct_desil1','pct_desil2','pct_belum_prmk',
        'skor_topsis_ahp','rank_topsis_ahp','rank_saw_ahp']
print('10 DESA PRIORITAS TERATAS (TOPSIS + bobot AHP):')
print(out.head(10)[show].round(4).to_string(index=False))

out.to_csv(SPK_DIR / 'ranking_prioritas_desa_AHP.csv', index=False)
bobot.to_csv(SPK_DIR / 'bobot_ahp.csv', index=False)

import json
with open(SPK_DIR / 'ahp_ringkasan.json', 'w', encoding='utf-8') as f:
    json.dump({
        'lambda_max': round(float(lamda_max),4), 'CI': round(float(CI),4),
        'RI': RI, 'CR': round(float(CR),4), 'konsisten': bool(CR < 0.10),
        'bobot_ahp': {c: round(float(wi),4) for c,wi in zip(KRITERIA, w)},
        'korelasi_topsis_ahp_vs_ilustrasi': round(float(rho_bobot),4),
        'korelasi_topsis_vs_saw_bobotAHP': round(float(rho_metode),4),
        'catatan': 'Perbandingan berpasangan adalah template; wajib divalidasi Dinas Sosial.',
    }, f, indent=2, ensure_ascii=False)
print('\nHasil disimpan di:', SPK_DIR)

10 DESA PRIORITAS TERATAS (TOPSIS + bobot AHP):
    KECAMATAN         KELURAHAN  cluster_kmeans  pct_desil1  pct_desil2  pct_belum_prmk  skor_topsis_ahp  rank_topsis_ahp  rank_saw_ahp
Kikim Selatan  Beringin Janggut               0     44.3662     13.3803          4.9296           0.6134                1             2
  Muarapayang       Bandu Agung               0     39.4366     18.7793          4.2254           0.6006                2             1
        Jarai       Lubuk Saung               0     30.5755     15.8273          5.3957           0.5434                3            10
 Tanjungtebat     Padang Perigi               0     30.2905     19.9170          4.1494           0.5204                4            13
Kikim Selatan        Keban Raya               0     24.0506     18.9873          6.3291           0.5038                5             7
  Kikim Timur    Muara Empayang               0     28.5714     17.4603          4.7619           0.5035                6             9


## 7. Ringkasan

- AHP menghasilkan bobot kriteria yang **teruji konsistensinya** (CR < 0,10) menggantikan
  bobot ilustrasi.
- Bobot AHP diterapkan ke TOPSIS & SAW; hasil ranking dibandingkan dengan bobot sebelumnya.
- Output: `bobot_ahp.csv`, `ranking_prioritas_desa_AHP.csv`, `ahp_ringkasan.json`.
- **Langkah validasi wajib:** isi ulang matriks perbandingan berpasangan bersama Dinas Sosial
  Kab. Lahat agar bobot benar-benar merepresentasikan pertimbangan ahli.